# Course research · Run All on Kaggle T4 ×2

Upload this notebook and the **small research dataset ZIP**, attach the dataset,
complete one setup cell, select **T4 ×2**, turn Internet **On**, then **Run All**.
The notebook downloads the pinned dependencies and original model checkpoint,
verifies their native hashes and versions, and advances automatically:

**Hardware → first 90 slots → full pilot → audit and freeze → complete main matrix
→ independently reproduced analysis ? results and resume ZIPs.**

Progress appears every 15 seconds; portable backups update every ten minutes.
No action switching, separate asset upload, or manual stage restart is needed.

**A complete result depends on every mandatory gate passing and enough session time.**
Failed gates finish with a limited result and preserved evidence. Prior spending
and the supported complete memory bound cannot be invented. The default permits
CPU reporting only; record separate conditional GPU, freeze and main authorizations
below. Use Accelerator **None** for CPU reporting. GPU work requires the pinned
Linux x86_64 **Python 3.13** candidate; incompatible images stop safely.

Keep the dataset private. Later sessions also attach **course-resume.zip**;
Run All resumes exact saved inputs without resampling completed calls.
For durable background execution, use **Save Version → Save & Run All**;
see the [Kaggle notebook guide](https://www.kaggle.com/docs/notebooks).


In [1]:
# ONE SETUP CELL. The repo chooses the next stage automatically.
ACTION = "run_all"   # advanced read-only alternatives: inspect, status, analyze, export
ENABLE_GPU = False
PACKAGE_DIR = None   # detects the exact attached dataset automatically
RESUME_ZIP = None    # detects the single attached course-resume.zip automatically
ASSETS_ZIP = None    # optional existing offline assets; online downloads are the default

# Current provider facts reset on every resume. Empty dictionaries retain saved
# authorizations, base bounds and reviews. Unknown duration is never zero.
RUN_SETTINGS = {
    "allocated_at": None,                 # actual UTC assignment, "...+00:00"
    "provider_quota_gpu_hours": None,      # allowance verified at this assignment
    "provider_session_seconds": None,      # verified maximum duration
    "prior_sessions_released": False,
    "authorizations": {
        # Add five separate scope statements once:
        # weights="I authorize downloading the original pinned checkpoint and dependencies...",
        # hardware="I authorize the bounded course hardware screen...",
        # pilot="I authorize the fixed five-world development pilot...",
        # freeze="I authorize one unchanged freeze conditional on all gates...",
        # main="I authorize the fixed 20-world matrix within course/overall caps..."
    },
    "budget": {
        # prior_development_gpu_hours=[lower,upper], prior_setup_gpu_hours=[...],
        # prior_interruption_gpu_hours=[...], prior_other_course_gpu_hours=[...],
        # prior_other_overall_gpu_hours=[...], prior_smoke_gpu_hours=[...],
        # prior_smoke_attempts=..., prior_evidence="..."
        # Keep the SAME base values on every resume. Supported zero needs evidence.
    },
    "hardware_review": {
        # accepted=True, complete_live_memory_upper_bound_bytes=...,
        # justification="supported complete allocator/workspace/load bound..."
        # Stress-test success alone does not establish this required proof.
    },
    "previous_session_release": {
        # released_at="...+00:00", evidence="actual provider deallocation receipt"
    },
}


In [2]:
import hashlib,json,os,shutil,stat,sys,time,zipfile
from pathlib import Path,PurePosixPath
from IPython.display import display,FileLink,HTML
EXPECTED_MANIFEST_SHA256='0137dfa544bd95303af0b4a1c63db5a9524786170498f5aa5f77d71d8a29df30'
EXPECTED_SOURCE_COMMIT='27cc3fce4b5b8df111eb8e2904e77705b8a06c38'
workspace=Path(os.environ.get('COURSE_WORKSPACE','/kaggle/working/course-workspace')).resolve()
workspace.mkdir(parents=True,exist_ok=True)
bundle=workspace/'bundle'
inputs=Path(os.environ.get('COURSE_INPUT','/kaggle/input'))
def attached(name,explicit=None):
    if explicit: return Path(explicit).resolve()
    matches=list(inputs.rglob('course-resume*.zip' if name=='course-resume.zip' else name))
    if len(matches)>1: raise ValueError('Select one exact '+name+' in the options cell')
    return matches[0] if matches else None
def extract_fresh(archive,target):
    if target.exists(): raise FileExistsError(target)
    with zipfile.ZipFile(archive) as z:
        names=set()
        for i in z.infolist():
            p=PurePosixPath(i.filename)
            if p.is_absolute() or '..' in p.parts or '\\' in i.filename or ':' in i.filename or i.filename in names or stat.S_ISLNK(i.external_attr>>16):
                raise ValueError('Unsafe/duplicated archive member')
            names.add(i.filename)
        target.mkdir(parents=True)
        z.extractall(target)
resume=attached('course-resume.zip',RESUME_ZIP)
if resume is None:
    # Kaggle normally expands uploaded ZIPs. Recognize the extracted original
    # bundle instead of accidentally starting a new set of worker files.
    options=[p.parent for p in inputs.rglob('operator-config.json')
        if (p.parent/'inputs/population.json').is_file()
        and (p.parent/'source/source-manifest.json').is_file()
        and hashlib.sha256((p.parent/'source/source-manifest.json').read_bytes()).hexdigest()==EXPECTED_MANIFEST_SHA256]
    if len(options)>1: raise ValueError('Select the exact extracted resume directory with RESUME_ZIP')
    resume=options[0] if options else None
if not bundle.exists():
    if resume and resume.is_dir():
        members=list(resume.rglob('*'))
        if any(p.is_symlink() or p.name.endswith(('-wal','-shm','.owner','.tmp')) for p in members):
            raise ValueError('Resume requires the ordinary exported SQLite backups, without live WAL/locks or symlinks')
        shutil.copytree(resume,bundle)
    elif resume: extract_fresh(resume,bundle)
    else: bundle.mkdir()
source=bundle/'source'
if not source.exists():
    options=[Path(PACKAGE_DIR)/'source-manifest.json'] if PACKAGE_DIR else list(inputs.rglob('source-manifest.json'))
    matches=[p for p in options if hashlib.sha256(p.read_bytes()).hexdigest()==EXPECTED_MANIFEST_SHA256]
    if len(matches)!=1: raise ValueError('Attach the exact course dataset ZIP (Kaggle extracts it), or set PACKAGE_DIR')
    package=matches[0].parent
    manifest=json.loads(matches[0].read_bytes())
    source.mkdir()
    for name,expected in manifest['members'].items():
        p=PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name: raise ValueError('Unsafe source member')
        original=package/name
        if not original.is_file() or original.is_symlink() or original.stat().st_size!=expected['bytes'] or hashlib.sha256(original.read_bytes()).hexdigest()!=expected['sha256']:
            raise ValueError('Source bytes differ: '+name)
        target=source/name
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(original,target)
    shutil.copyfile(matches[0],source/'source-manifest.json')
manifest_bytes=(source/'source-manifest.json').read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest()!=EXPECTED_MANIFEST_SHA256: raise ValueError('Resume requires the identical source package/notebook')
manifest=json.loads(manifest_bytes)
if (source/'SOURCE_COMMIT.txt').read_text().strip()!=EXPECTED_SOURCE_COMMIT: raise ValueError('Source commit differs')
for name,expected in manifest['members'].items():
    p=source/name
    if p.is_symlink() or not p.is_file() or p.stat().st_size!=expected['bytes'] or hashlib.sha256(p.read_bytes()).hexdigest()!=expected['sha256']:
        raise ValueError('Canonical source changed: '+name)
display(HTML('<div style="padding:16px;background:#e8f3f3;border-radius:10px"><b>Source verified</b> · '+EXPECTED_SOURCE_COMMIT[:12]+'<br>20 main worlds · 6,280 logical main uses · 5 separate development worlds</div>'))
display(FileLink(str(source/'docs/KAGGLE_COURSE.md')))


ValueError: Source bytes differ: notebooks/validation/course-m5-preparation/inputs/smoke-requests.json.gz

## Your saved configuration
Use `[lower, upper]` GPU-hour bounds supported by evidence. Zero also requires evidence. Preserve the same base bounds on every resume. Available account quota does not reset study spending.


In [3]:
# Automatic assembly; edit only the setup cell above.
saved_config=bundle/'operator-config.json'
CONFIG=json.loads((saved_config if saved_config.exists() else source/'configs/course-kaggle.json').read_bytes())
CONFIG.update(execute_gpu=ENABLE_GPU,continuous_assignment=(ACTION=='run_all'),
    automatic_pilot_review=True,max_work_seconds=36000,finalization_seconds=900)
for key,value in RUN_SETTINGS.items():
    if isinstance(value,dict): CONFIG.setdefault(key,{}).update(value)
    else: CONFIG[key]=value
config_path=bundle/'operator-config.json'
config_path.write_text(json.dumps(CONFIG,indent=2)+'\n')


1510

## Run and monitor
The package streams progress every 15 seconds and exports quiescent SQLite backups at ten-minute boundaries. Interrupted/invalid replies remain recorded. A failed gate stops subsequent work.


In [4]:
import queue,threading,subprocess,signal
def bounded_child(arguments,limit):
    # The repo owns scientific work; this cell only streams its process output.
    log=workspace/('action-'+time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'.log')
    process=subprocess.Popen([str(a) for a in arguments],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,bufsize=1,start_new_session=(os.name!='nt'),cwd=workspace,
        env=dict(os.environ,MPLBACKEND='Agg'))
    lines=queue.Queue()
    def collect():
        for line in process.stdout: lines.put(line)
    reader=threading.Thread(target=collect,daemon=True); reader.start()
    deadline=time.monotonic()+limit
    try:
        with log.open('w',encoding='utf-8') as output:
            while process.poll() is None or not lines.empty():
                if time.monotonic()>=deadline: raise TimeoutError('Notebook controller deadline; preserve/export and Stop Session')
                try: line=lines.get(timeout=1)
                except queue.Empty: continue
                output.write(line); output.flush(); print(line,end='',flush=True)
    finally:
        if process.poll() is None:
            if os.name!='nt': os.killpg(process.pid,signal.SIGTERM)
            else: process.terminate()
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                if os.name!='nt': os.killpg(process.pid,signal.SIGKILL)
                else: process.kill()
                process.wait(timeout=5)
        reader.join(timeout=2)
    return process.returncode

try:
    if ACTION=='inspect':
        print('Source verified. Default Run All automatically orchestrates the authorized stages.')
    else:
        gpu_action=ACTION in ('hardware','capability','pilot','main') or ACTION=='run_all' and ENABLE_GPU
        if gpu_action and not ENABLE_GPU: raise ValueError('GPU execution requires explicit scoped authorizations in the setup cell')
        assigned=Path('/proc/driver/nvidia/gpus').is_dir() and bool(list(Path('/proc/driver/nvidia/gpus').iterdir()))
        if assigned and ACTION=='run_all' and (not ENABLE_GPU or not CONFIG.get('allocated_at')):
            raise ValueError('Assigned GPUs are chargeable: complete the authorized setup cell, or select Accelerator None for CPU reporting')
        if assigned and ACTION!='run_all' and not gpu_action:
            raise ValueError('Use Accelerator None for standalone CPU actions; assigned idle time is chargeable')
        assets=attached('course-assets.zip',ASSETS_ZIP)
        if assets is None:
            options=[p.parent for p in inputs.rglob('prepared-assets.json') if (p.parent/'checkpoint').is_dir() and (p.parent/'wheels').is_dir()]
            if len(options)>1: raise ValueError('Select the exact asset directory with ASSETS_ZIP')
            assets=options[0] if options else None
        gpu_receipt=workspace/'gpu-environment'/'bootstrap'/'bootstrap.json'
        cpu_receipt=workspace/'cpu-environment'/'bootstrap'/'bootstrap.json'
        kind='setup' if ACTION=='setup' else 'gpu-environment' if gpu_action or gpu_receipt.exists() or assets and not cpu_receipt.exists() else 'cpu-environment'
        environment=workspace/kind
        receipt=environment/'bootstrap'/'bootstrap.json'
        if not receipt.exists():
            arguments=[sys.executable,'-I',source/'scripts/course_kaggle_bootstrap.py']
            if ACTION=='setup': arguments+=['setup','--config',config_path]
            elif kind=='gpu-environment':
                if assets is None: arguments+=['online','--config',config_path]
                else: arguments+=['restore','--assets',assets,'--config',config_path]
            else: arguments+=['cpu']
            arguments+=['--root',source,'--scratch',environment/'scratch','--output',environment/'bootstrap']
            if bounded_child(arguments,1800)!=0: raise RuntimeError('Bootstrap failed; its receipt and logs are retained')
        runtime=json.loads(receipt.read_bytes())
        if runtime.get('status')=='blocked': raise RuntimeError(runtime['reason'])
        # Preserve setup/hash/version evidence in every portable resume export.
        bootstrap_evidence=bundle/'environment-receipts'/kind
        bootstrap_evidence.mkdir(parents=True,exist_ok=True)
        for path in (environment/'bootstrap').glob('*'):
            if path.is_file() and path.name!='source-manifest.json':
                target=bootstrap_evidence/path.name
                if target.exists() and target.read_bytes()!=path.read_bytes():
                    target=bootstrap_evidence/(time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'-'+path.name)
                if not target.exists(): shutil.copyfile(path,target)
        if runtime.get('assets_receipt') and not runtime.get('requires_dataset_restore'):
            CONFIG.update(assets_receipt=runtime['assets_receipt'],hf_cache=runtime['hf_cache'])
            shutil.copyfile(runtime['assets_receipt'],bundle/'asset-receipt.json')
        config_path.write_text(json.dumps(CONFIG,indent=2)+'\n')
        if ACTION not in ('setup','restore'):
            arguments=[runtime['interpreter'],'-I',source/'scripts/course_kaggle_workflow.py',
                '--root',source,'--bundle',bundle,'--config',config_path,'--action',ACTION]
            limit=(CONFIG.get('provider_session_seconds') or CONFIG['max_work_seconds'])+2100 if gpu_action else 2100
            code=bounded_child(arguments,limit)
            print('Workflow returned',code,'; the final cell exports results and resume evidence.')
except (Exception,KeyboardInterrupt) as error:
    failure=dict(action=ACTION,status='blocked_or_safely_stopped',reason=type(error).__name__+': '+str(error),
        gates_bypassed=False,assignment_clock=CONFIG.get('allocated_at'),provider_release_confirmed=False)
    (bundle/'last-action.json').write_text(json.dumps(failure,indent=2)+'\n')
    if locals().get('environment') and (environment/'bootstrap').is_dir():
        evidence=bundle/'environment-receipts'/(kind+'-failed-'+time.strftime('%Y%m%dT%H%M%S',time.gmtime()))
        evidence.mkdir(parents=True,exist_ok=True)
        for path in (environment/'bootstrap').glob('*'):
            if path.is_file() and path.name!='source-manifest.json': shutil.copyfile(path,evidence/path.name)
    if locals().get('assigned'):
        try:
            ids=subprocess.check_output(['nvidia-smi','--query-gpu=uuid','--format=csv,noheader'],text=True,timeout=5)
            physical_devices=[line.strip() for line in ids.splitlines() if line.strip()]
        except Exception: physical_devices=[]
        old_run=json.loads((bundle/'run.json').read_bytes()) if (bundle/'run.json').exists() else None
        stage='main' if old_run and old_run['freeze'] else 'pilot' if old_run else 'hardware'
        receipt=dict(config=CONFIG,allocated_at=CONFIG.get('allocated_at'),physical_devices=physical_devices,
            stage=stage,reason=failure['reason'],observed_at=time.strftime('%Y-%m-%dT%H:%M:%S+00:00',time.gmtime()))
        refusals=bundle/'bootstrap-refusals'; refusals.mkdir(exist_ok=True)
        (refusals/(time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'.json')).write_text(json.dumps(receipt,indent=2)+'\n')
    print(failure['reason'])



ValueError: Assigned GPUs are chargeable: complete the authorized setup cell, or select Accelerator None for CPU reporting


## Results and safe resume
The cell below always runs after an orchestration stop. Download both ZIPs and the executed notebook. A later Run All resumes the same saved inputs; actual provider release still needs a receipt.


In [5]:
# Run this cell after an action finishes/fails. It never invokes inference.
# Stdlib fallback also exports setup/refusal evidence before project installation.
import sqlite3,tempfile
from contextlib import closing
# A small results export accompanies the complete reproducibility/resume ZIP.
import html
last=json.loads((bundle/'last-action.json').read_bytes()) if (bundle/'last-action.json').exists() else {}
analysis=json.loads((bundle/'analysis-receipt.json').read_bytes()) if (bundle/'analysis-receipt.json').exists() else {}
status=analysis.get('status') or last.get('status','limited_result')
reason=last.get('reason') or last.get('stop_reason') or ''
coverage=analysis.get('coverage')
summary='<html><meta charset="utf-8"><title>Course research result</title><body style="font:16px system-ui;max-width:900px;margin:40px auto;line-height:1.5">'
summary+='<h1>Course research result</h1><p><b>'+html.escape(status)+'</b></p><p>'+html.escape(reason)+'</p>'
if coverage: summary+='<p>Fixed main population coverage: '+html.escape(json.dumps(coverage))+'</p>'
summary+='<p>Original replies and failures are retained in course-resume.zip. Unexecuted outcomes remain unknown. No failed gate was bypassed.</p>'
if analysis.get('report'): summary+='<p><a href="'+html.escape(analysis['report'],quote=True)+'">Open the independently reproduced analysis</a></p>'
summary+='<p>Retain both ZIPs and this executed notebook. Stop any interactive GPU session and retain its real release receipt.</p></body></html>'
(bundle/'RESULT.html').write_text(summary,encoding='utf-8')
resume_export=workspace/'course-resume.zip'
partial=resume_export.with_suffix('.tmp')
with tempfile.TemporaryDirectory() as temporary:
    with zipfile.ZipFile(partial,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for p in sorted(bundle.rglob('*')):
            if not p.is_file() or p.is_symlink() or p.name.endswith(('-wal','-shm','.owner','.tmp','.pyc')) or '__pycache__' in p.parts: continue
            target=p
            if p.name in ('worker-0.sqlite','worker-1.sqlite'):
                target=Path(temporary)/(p.parent.name+'-'+p.name)
                with closing(sqlite3.connect(p.as_uri()+'?mode=ro',uri=True)) as original,closing(sqlite3.connect(target)) as copy:
                    original.backup(copy)
                    if copy.execute('PRAGMA integrity_check').fetchone()[0]!='ok': raise ValueError('SQLite backup failed')
            z.write(target,p.relative_to(bundle).as_posix())
try:
    partial.replace(resume_export)
except PermissionError:
    # An open download/Windows scanner may hold the previous derived ZIP.
    # Keep it intact and publish this complete snapshot under a fresh name.
    resume_export=workspace/('course-resume-'+time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'.zip')
    partial.replace(resume_export)
results_export=workspace/'course-results.zip'
with zipfile.ZipFile(results_export,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in sorted(bundle.rglob('*')):
        if p.is_file() and not p.is_symlink() and (p.suffix in ('.html','.csv','.svg','.png') or
                p.parent==bundle and p.suffix=='.json' and p.name!='operator-config.json'):
            z.write(p,p.relative_to(bundle).as_posix())
display(HTML(summary))
for p in (results_export,resume_export,workspace/'setup'/'course-assets.zip'):
    if p.exists(): display(FileLink(str(p)))
if (bundle/'last-action.json').exists():
    display(FileLink(str(bundle/'last-action.json')))
if (bundle/'analysis-receipt.json').exists():
    analysis=json.loads((bundle/'analysis-receipt.json').read_bytes())
    display(FileLink(str(bundle/analysis['report'])))
    print(analysis['status'],analysis['coverage'])
print('Retain the ZIP(s) and actual executed notebook. For GPU actions: Stop Session now and verify provider deallocation. Record its UTC release and evidence for the next session.')


/kaggle/working/course-workspace/course-results.zip

/kaggle/working/course-workspace/course-resume.zip

/kaggle/working/course-workspace/bundle/last-action.json

Retain the ZIP(s) and actual executed notebook. For GPU actions: Stop Session now and verify provider deallocation. Record its UTC release and evidence for the next session.
